# Préparation des datasets d'entraînement

**Objectif** : construire les datasets qui serviront à la modélisation, avec les bonnes lignes et les bonnes variables, sans fuite de données.

| Service | Source | Cible |
|---|---|---|
| `/predict` | `data/agriculture-crop-yield/crop_yield.csv` | `Yield_tons_per_hectare` |
| `/recommend` | `data/processed/crop_yield_clean.csv` (notebook 04) | `yield_t_ha` |

Pas d'encodage, de standardisation ni d'imputation ici : ces étapes seront apprises sur le train
pendant la modélisation.

## Imports

In [1]:
import numpy as np
import pandas as pd

from agritech.config import AGRICULTURE_CROP_YIELD_FILENAME, PATHS, SEED

# 1. `/predict` — Agriculture CropYield

## Lecture et contrôles de base

In [2]:
csv_path = PATHS.data_agriculture_crop_yield / AGRICULTURE_CROP_YIELD_FILENAME
df_agri = pd.read_csv(csv_path)

print("lignes x colonnes  :", df_agri.shape)
print("valeurs manquantes :", df_agri.isna().sum().sum())
print("doublons stricts   :", df_agri.duplicated().sum())
print("\ntypes :")
print(df_agri.dtypes.to_string())

lignes x colonnes  : (1000000, 10)
valeurs manquantes : 0
doublons stricts   : 0

types :
Region                     object
Soil_Type                  object
Crop                       object
Rainfall_mm               float64
Temperature_Celsius       float64
Fertilizer_Used              bool
Irrigation_Used              bool
Weather_Condition          object
Days_to_Harvest             int64
Yield_tons_per_hectare    float64


**Observations :**

- Aucune valeur manquante, aucun doublon, types corrects.

## Cible : rendements négatifs

Un rendement négatif est impossible. Les autres valeurs atypiques sont gardées.

In [3]:
negatifs = df_agri[df_agri["Yield_tons_per_hectare"] < 0].copy()
print(f"rendements négatifs exclus : {len(negatifs)} ({len(negatifs) / len(df_agri):.3%})")

rendements négatifs exclus : 231 (0.023%)


**Observations :**

- Les 231 rendements négatifs sont exclus de l’entraînement et sauvegardés à part pour un test exploratoire après modélisation.
- Leurs cibles invalides ne serviront pas à évaluer les performances.

## Variables conservées pour la modélisation

Les neuf variables d’entrée sont conservées. Leur utilité et leur disponibilité au moment de prédire seront examinées lors de la modélisation.

In [4]:
FEATURES_PREDICT = [
    "Crop",
    "Soil_Type",
    "Rainfall_mm",
    "Temperature_Celsius",
    "Fertilizer_Used",
    "Irrigation_Used",
    "Region",
    "Weather_Condition",
    "Days_to_Harvest",
]
CIBLE_PREDICT = "Yield_tons_per_hectare"

df_predict = df_agri[df_agri[CIBLE_PREDICT] >= 0]

X_predict = df_predict[FEATURES_PREDICT]
y_predict = df_predict[CIBLE_PREDICT]

print(f"lignes           : {len(X_predict)} (sur {len(df_agri)}, soit {len(df_agri) - len(X_predict)} retirées)")
print(f"features         : {FEATURES_PREDICT}")
print(f"cible            : {CIBLE_PREDICT}")
print(f"manquants dans X : {X_predict.isna().sum().sum()}")
X_predict.head()

lignes           : 999769 (sur 1000000, soit 231 retirées)
features         : ['Crop', 'Soil_Type', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used', 'Region', 'Weather_Condition', 'Days_to_Harvest']
cible            : Yield_tons_per_hectare
manquants dans X : 0


,Crop,Soil_Type,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Region,Weather_Condition,Days_to_Harvest
0,Cotton,Sandy,897.077239,27.676966,False,True,West,Cloudy,122
1,Rice,Clay,992.673282,18.026142,True,True,South,Rainy,140
2,Barley,Loam,147.998025,29.794042,False,False,North,Sunny,106
3,Soybean,Sandy,986.866331,16.644190,False,True,North,Rainy,146
4,Wheat,Silt,730.379174,31.620687,True,True,South,Cloudy,110


**Observations :**

- 999 769 lignes, 9 variables d’entrée, aucune valeur manquante.
- Les variables catégorielles restent en texte : l’encodage sera appris sur le train.

# 2. `/recommend` — dataset historique nettoyé

Fonctionnement prévu : l'utilisateur choisit son pays, l'application préremplit la température, la
pluie et les pesticides avec les valeurs historiques de ce pays, et l'utilisateur peut les modifier.
Le modèle prédit ensuite le rendement des 10 cultures. Le pays sert seulement à préremplir : ce
n'est pas une variable du modèle.

Organisation dans le temps : l'application est pensée pour 2014, avec les données connues jusqu'en
2013. Pour une année *t*, les features n'utilisent donc que les années avant *t*. 2013 est gardée
pour le test final.

## Lecture et contrôles de base

In [5]:
CLE = ["iso3", "year", "crop"]
df_hist = pd.read_csv(PATHS.data_processed / "crop_yield_clean.csv")

print("lignes x colonnes   :", df_hist.shape)
print("doublons sur la clé :", df_hist.duplicated(CLE).sum())
print("cultures / pays     :", df_hist.crop.nunique(), "/", df_hist.iso3.nunique())
print("années              :", df_hist.year.min(), "-", df_hist.year.max())
print("valeurs manquantes  :", df_hist.isna().sum().sum())
print("rendements <= 0     :", (df_hist.yield_t_ha <= 0).sum())

assert len(df_hist) == 16_357
assert not df_hist.duplicated(CLE).any()
assert df_hist.isna().sum().sum() == 0

lignes x colonnes   : (16357, 8)
doublons sur la clé : 0
cultures / pays     : 10 / 117
années              : 1990 - 2013
valeurs manquantes  : 0
rendements <= 0     : 0


**Observations :**

- `crop_yield_clean.csv` : 16 357 lignes, 117 pays, 10 cultures, 1990-2013.
- Aucune valeur manquante, aucun rendement nul ou négatif.
- Le nettoyage a été fait dans le notebook 04 : il n'est pas refait ici.

## Features historiques

Pour chaque pays et chaque année *t*, on calcule la moyenne des 3 années précédentes (ou moins s'il
n'y en a pas 3). L'année *t* n'est jamais utilisée : `shift(1)` décale les valeurs d'un an, puis
`rolling(3)` calcule la moyenne glissante. Pour 2013, la moyenne porte sur 2010-2012.

- `temp_hist` : moyenne de la température des années *t*−3 à *t*−1.
- `log_pest_hist` : moyenne des pesticides des années *t*−3 à *t*−1, puis `log1p`, car les tonnages
  sont très déséquilibrés entre pays.
- `rain_mm` : valeur de pluie fixe par pays dans le fichier, identique chaque année, donc sans décalage.

In [6]:
def moyenne_historique(serie):
    """Moyenne des 3 années précédentes au plus, sans l'année courante."""
    return serie.shift(1).rolling(3, min_periods=1).mean()


# une ligne par pays et par année, triée par année : shift(1) prend bien l'année précédente
contexte_pays = (
    df_hist[["iso3", "year", "avg_temp", "pesticides_t"]]
    .drop_duplicates(["iso3", "year"])
    .sort_values(["iso3", "year"])
)
contexte_pays["temp_hist"] = contexte_pays.groupby("iso3")["avg_temp"].transform(moyenne_historique)
contexte_pays["pest_hist"] = contexte_pays.groupby("iso3")["pesticides_t"].transform(moyenne_historique)
assert contexte_pays["pesticides_t"].ge(0).all()
contexte_pays["log_pest_hist"] = np.log1p(contexte_pays["pest_hist"])

assert np.isfinite(contexte_pays["log_pest_hist"].dropna()).all()

df_hist = df_hist.merge(
    contexte_pays[["iso3", "year", "temp_hist", "pest_hist", "log_pest_hist"]],
    on=["iso3", "year"],
    how="left",
)
print("lignes après jointure :", len(df_hist))
print("colonnes              :", df_hist.columns.tolist())

lignes après jointure : 16357
colonnes              : ['iso3', 'area', 'year', 'crop', 'yield_t_ha', 'avg_temp', 'rain_mm', 'pesticides_t', 'temp_hist', 'pest_hist', 'log_pest_hist']


In [7]:
exemple = contexte_pays[contexte_pays["iso3"] == "FRA"]
annees = [1990, 1991, 1992, 1993, 2010, 2011, 2012, 2013]
colonnes = ["year", "avg_temp", "temp_hist", "pesticides_t", "pest_hist", "log_pest_hist"]

print("France :")
print(exemple[exemple["year"].isin(annees)][colonnes].round(2).to_string(index=False))

France :
 year  avg_temp  temp_hist  pesticides_t  pest_hist  log_pest_hist
 1990     11.96        NaN      97701.00        NaN            NaN
 1991     10.60      11.96     103434.00   97701.00          11.49
 1992     11.15      11.28      85249.00  100567.50          11.52
 1993     10.70      11.24      91953.00   95461.33          11.47
 2010     10.41      11.50      61903.00   73169.33          11.20
 2011     12.33      11.05      61039.00   68052.00          11.13
 2012     11.22      11.40      63547.59   62206.00          11.04
 2013     11.01      11.32      66497.29   62163.20          11.04


### Contrôles contre la fuite de données

On vérifie que les features n'utilisent pas l'année en cours : les années se suivent pour chaque
pays, la première année n'a pas d'historique, et la moyenne est recalculée à la main sur 300 lignes
pour `temp_hist` et `pest_hist`.

In [8]:
consecutives = contexte_pays.groupby("iso3")["year"].diff().dropna().eq(1).all()
print("années consécutives par pays         :", consecutives)

premieres = contexte_pays.groupby("iso3").head(1)
premiere_vide = premieres["temp_hist"].isna().all() and premieres["pest_hist"].isna().all()
print("features vides sur la première année :", premiere_vide)

# recalcul à la main de la moyenne t-3..t-1 sur 300 couples pays-année tirés au hasard
ecart_max = {"temp_hist": 0.0, "pest_hist": 0.0}
for _, ligne in contexte_pays.dropna(subset=["temp_hist"]).sample(300, random_state=SEED).iterrows():
    fenetre = contexte_pays[
        (contexte_pays["iso3"] == ligne["iso3"])
        & contexte_pays["year"].between(ligne["year"] - 3, ligne["year"] - 1)
    ]
    ecart_max["temp_hist"] = max(ecart_max["temp_hist"], abs(fenetre["avg_temp"].mean() - ligne["temp_hist"]))
    ecart_max["pest_hist"] = max(ecart_max["pest_hist"], abs(fenetre["pesticides_t"].mean() - ligne["pest_hist"]))
print("écart max avec la moyenne t-3..t-1 recalculée :", {nom: round(e, 6) for nom, e in ecart_max.items()})

assert consecutives and premiere_vide
assert max(ecart_max.values()) < 1e-6

années consécutives par pays         : True
features vides sur la première année : True
écart max avec la moyenne t-3..t-1 recalculée : {'temp_hist': np.float64(0.0), 'pest_hist': np.float64(0.0)}


**Observations :**

- Le recalcul à la main donne exactement les mêmes valeurs pour `temp_hist` et `pest_hist` :
  l'année *t* n'est pas utilisée.
- Exemple de la France : la valeur de 2013 est la moyenne de 2010-2012, et 1990 n'a pas
  d'historique.

## Pourquoi 693 lignes disparaissent

La première année d'un pays n'a pas d'année précédente : ses features historiques sont vides. Ces
lignes ne peuvent pas servir à l'entraînement.

In [9]:
df_recommend = df_hist.dropna(subset=["temp_hist", "rain_mm", "log_pest_hist"])
perdues = df_hist.drop(index=df_recommend.index)

premiere_annee = df_hist.groupby("iso3")["year"].transform("min")
lignes_premiere_annee = df_hist.index[df_hist["year"] == premiere_annee]

print(f"dataset historique nettoyé : {len(df_hist)} lignes")
print(f"dataset d'entraînement     : {len(df_recommend)} lignes")
print(f"lignes retirées            : {len(perdues)}")
print("lignes retirées = première année de chaque pays :", perdues.index.equals(lignes_premiere_annee))

assert len(perdues) == 693
assert perdues.index.equals(lignes_premiere_annee)

dataset historique nettoyé : 16357 lignes
dataset d'entraînement     : 15664 lignes
lignes retirées            : 693
lignes retirées = première année de chaque pays : True


In [10]:
# Années et pays concernés
print(perdues.groupby("year").agg(lignes=("crop", "size"), pays=("iso3", "nunique")).to_string())

print()
for annee, groupe in perdues[perdues["year"] > df_hist["year"].min()].groupby("year"):
    print(f"{annee} : {', '.join(sorted(groupe['area'].unique()))}")

      lignes  pays
year              
1990     605    97
1992      62    14
1993      13     3
2000       3     1
2006       3     1
2012       7     1

1992 : Armenia, Azerbaijan, Belarus, Croatia, Estonia, Kazakhstan, Latvia, Lithuania, Republic of Moldova, Russian Federation, Slovenia, Tajikistan, The former Yugoslav Republic of Macedonia, Ukraine
1993 : Czechia, Eritrea, Slovakia
2000 : Belgium
2006 : Montenegro
2012 : Sudan


In [11]:
# État après jointures : non sauvegardé, ses chiffres sont vérifiés par assert dans le notebook 04
APRES_JOINTURES = {"lignes": 22_679, "pays": 168, "période": "1990-2013"}

chaine = pd.DataFrame(
    {
        "lignes": [APRES_JOINTURES["lignes"], len(df_hist), len(df_recommend)],
        "pays": [APRES_JOINTURES["pays"], df_hist["iso3"].nunique(), df_recommend["iso3"].nunique()],
        "cultures": [10, df_hist["crop"].nunique(), df_recommend["crop"].nunique()],
        "période": [
            APRES_JOINTURES["période"],
            f"{df_hist['year'].min()}-{df_hist['year'].max()}",
            f"{df_recommend['year'].min()}-{df_recommend['year'].max()}",
        ],
    },
    index=["état après jointures", "dataset historique nettoyé", "dataset d'entraînement /recommend"],
)
chaine["lignes retirées"] = (-chaine["lignes"].diff()).fillna(0).astype(int)
chaine["pays retirés"] = (-chaine["pays"].diff()).fillna(0).astype(int)

assert chaine["lignes retirées"].tolist() == [0, 6_322, 693]
assert chaine["pays retirés"].tolist() == [0, 51, 0]
assert (df_recommend["year"].min(), df_recommend["year"].max()) == (1991, 2013)
print("rendements <= 0 dans le dataset d'entraînement :", (df_recommend["yield_t_ha"] <= 0).sum())
chaine

rendements <= 0 dans le dataset d'entraînement : 0


,lignes,pays,cultures,période,lignes retirées,pays retirés
état après jointures,22679,168,10,1990-2013,0,0
dataset historique nettoyé,16357,117,10,1990-2013,6322,51
dataset d'entraînement /recommend,15664,117,10,1991-2013,693,0


**Observations :**

- Les 693 lignes retirées sont exactement celles de la première année de chaque pays.
- Cette première année est 1990 pour 97 pays. Pour les autres, c'est l'année où le pays apparaît
  dans les données : 1992 pour l'ex-URSS et l'ex-Yougoslavie, 1993 pour la Tchéquie, la Slovaquie
  et l'Érythrée, 2000 pour la Belgique, 2006 pour le Monténégro, 2012 pour le Soudan.
- Le dataset d'entraînement commence donc en 1991.
- Résumé : 22 679 lignes après les jointures → 16 357 après le nettoyage → 15 664 pour `/recommend`.

## Les 10 cultures

Nombre de lignes, de pays et d'années par culture dans le dataset d'entraînement.

In [12]:
couverture_cultures = df_recommend.groupby("crop").agg(
    lignes=("yield_t_ha", "size"),
    pays=("iso3", "nunique"),
    années=("year", "nunique"),
    rendement_médian=("yield_t_ha", "median"),
).sort_values("lignes", ascending=False)

couverture_cultures.round(2)

,lignes,pays,années,rendement_médian
crop,,,,
Maize,2407,109,23,2.60
Potatoes,2401,109,23,15.95
Wheat,2095,95,23,2.48
"Rice, paddy",1815,82,23,3.49
Sorghum,1713,80,23,1.27
Soybeans,1589,73,23,1.60
Sweet potatoes,1368,61,23,8.25
Cassava,1127,49,23,10.00
Plantains and others,602,27,23,8.40


**Observations :**

- Les 10 cultures sont présentes sur les 23 années.
- L'igname (547 lignes, 25 pays) et le plantain (602 lignes, 27 pays) ont le moins de données.
- Rendement médian de 1,3 t/ha (sorgho) à 16 t/ha (pomme de terre) : les tubercules arriveront en
  tête d'un classement en t/ha.

## Conditions observées avant 2013

Quantiles 5 % et 95 % calculés hors test final, sur les années avant 2013. Ces repères décrivent les données, pas la fiabilité du modèle. Pour une règle évaluée en validation, ils devront être recalculés sur le train de chaque découpage.

In [13]:
donnees_avant_test = df_recommend[df_recommend["year"] < 2013]
domaine = donnees_avant_test.groupby("crop").agg(
    temp_p5=("temp_hist", lambda s: s.quantile(0.05)),
    temp_p95=("temp_hist", lambda s: s.quantile(0.95)),
    pluie_p5=("rain_mm", lambda s: s.quantile(0.05)),
    pluie_p95=("rain_mm", lambda s: s.quantile(0.95)),
)

domaine.sort_values("temp_p5").round(0)

,temp_p5,temp_p95,pluie_p5,pluie_p95
crop,,,,
Wheat,6.0,27.0,89.0,1996.0
Potatoes,6.0,28.0,92.0,2274.0
Maize,8.0,28.0,92.0,2387.0
Soybeans,8.0,27.0,250.0,2280.0
Sorghum,9.0,28.0,92.0,2280.0
"Rice, paddy",9.0,28.0,151.0,2702.0
Sweet potatoes,11.0,28.0,190.0,2691.0
Yams,16.0,28.0,282.0,3142.0
Plantains and others,17.0,28.0,1071.0,2387.0


**Observations :**

- Ces intervalles couvrent les 90 % centraux de chaque variable, par culture, avant 2013.
- Une valeur extérieure est inhabituelle dans ces données, sans prouver que la prédiction est peu fiable.

## Variables candidates

| Variable | Rôle | Pourquoi |
|---|---|---|
| `crop` | variable candidate | culture à classer |
| `temp_hist` | variable candidate | température moyenne des années précédentes |
| `rain_mm` | variable candidate | valeur de pluie fixe par pays dans le fichier |
| `pest_hist` | variable candidate | moyenne des pesticides des années précédentes, en tonnes |
| `log_pest_hist` | variable candidate | logarithme de cette moyenne, à comparer à la version brute |
| `avg_temp` | non gardée | remplacée par `temp_hist` : la température de l'année en cours n'est pas connue au moment de la prédiction |
| `pesticides_t` | non gardée | remplacée par les variables historiques, pour la même raison |
| `iso3`, `area` | hors modèle | préremplir les valeurs du pays, analyses par pays |
| `year` | hors modèle | séparer les années : avant 2013 pour l'entraînement et la validation, 2013 pour le test final |

Les versions brute et logarithmique sont conservées pour comparaison, pas imposées ensemble au modèle.

In [14]:
FEATURES_RECOMMEND = ["crop", "temp_hist", "rain_mm", "pest_hist", "log_pest_hist"]
CIBLE_RECOMMEND = "yield_t_ha"
CONTEXTE_RECOMMEND = ["iso3", "area", "year"]   # hors modèle

X_recommend = df_recommend[FEATURES_RECOMMEND]
y_recommend = df_recommend[CIBLE_RECOMMEND]
contexte_recommend = df_recommend[CONTEXTE_RECOMMEND]

print(f"lignes               : {len(X_recommend)}")
print(f"features             : {FEATURES_RECOMMEND}")
print("remplacées           : ['avg_temp', 'pesticides_t']")
print(f"contexte hors modèle : {CONTEXTE_RECOMMEND}")
print(f"cible                : {CIBLE_RECOMMEND}")
print(f"manquants dans X     : {X_recommend.isna().sum().sum()}")
X_recommend.head()

lignes               : 15664
features             : ['crop', 'temp_hist', 'rain_mm', 'pest_hist', 'log_pest_hist']
remplacées           : ['avg_temp', 'pesticides_t']
contexte hors modèle : ['iso3', 'area', 'year']
cible                : yield_t_ha
manquants dans X     : 0


,crop,temp_hist,rain_mm,pest_hist,log_pest_hist
1,Maize,16.370000,1485.0,121.000000,4.804021
2,Maize,15.865000,1485.0,121.000000,4.804021
3,Maize,15.930000,1485.0,121.000000,4.804021
4,Maize,15.823333,1485.0,121.000000,4.804021
5,Maize,16.356667,1485.0,147.666667,5.001707


**Observations :**

- 15 664 lignes, 5 variables candidates, dont deux versions des pesticides, aucune valeur manquante.
- `iso3`, `area` et `year` restent dans le fichier, mais ne sont pas des features.

# 3. Sauvegarde

Deux datasets d’entraînement et un fichier de rendements négatifs dans `data/processed/`, non versionnés et reconstruits par ce notebook. Pas d'encodage
ni de standardisation : ces étapes seront apprises sur le train pendant la modélisation. Le fichier
`/recommend` garde `iso3`, `area` et `year` pour préremplir les valeurs du pays, séparer les années
et faire des analyses.
Le fichier des rendements négatifs est réservé à l’examen des prédictions, hors entraînement et évaluation.

In [15]:
PATHS.data_processed.mkdir(parents=True, exist_ok=True)
chemin_predict = PATHS.data_processed / "predict_training_dataset.csv"
chemin_recommend = PATHS.data_processed / "recommend_training_dataset.csv"

df_predict[FEATURES_PREDICT + [CIBLE_PREDICT]].to_csv(chemin_predict, index=False)
df_recommend[CONTEXTE_RECOMMEND + FEATURES_RECOMMEND + [CIBLE_RECOMMEND]].to_csv(chemin_recommend, index=False)

chemin_negatifs = PATHS.data_processed / "predict_negative_yield_rows.csv"
negatifs.to_csv(chemin_negatifs, index=False)
print("écrit :", chemin_negatifs.relative_to(PATHS.root))

# chemins relatifs : pas de chemin local enregistré dans le notebook
print("écrit :", chemin_predict.relative_to(PATHS.root))
print("écrit :", chemin_recommend.relative_to(PATHS.root))

écrit : data/processed/predict_negative_yield_rows.csv
écrit : data/processed/predict_training_dataset.csv
écrit : data/processed/recommend_training_dataset.csv


In [16]:
# Relecture des deux fichiers et contrôles
relu_predict = pd.read_csv(chemin_predict)
relu_recommend = pd.read_csv(chemin_recommend)

assert len(relu_predict) == 999_769
assert relu_predict.columns.tolist() == FEATURES_PREDICT + [CIBLE_PREDICT]
assert relu_predict.isna().sum().sum() == 0
assert (relu_predict[CIBLE_PREDICT] >= 0).all()

assert len(relu_recommend) == 15_664
assert relu_recommend.columns.tolist() == CONTEXTE_RECOMMEND + FEATURES_RECOMMEND + [CIBLE_RECOMMEND]
assert relu_recommend["iso3"].nunique() == 117
assert relu_recommend["crop"].nunique() == 10
assert (relu_recommend["year"].min(), relu_recommend["year"].max()) == (1991, 2013)
assert not relu_recommend.duplicated(["iso3", "year", "crop"]).any()
assert relu_recommend[FEATURES_RECOMMEND + [CIBLE_RECOMMEND]].isna().sum().sum() == 0

print("predict   :", relu_predict.shape, "| colonnes :", relu_predict.columns.tolist())
print("recommend :", relu_recommend.shape, "| colonnes :", relu_recommend.columns.tolist())
print(f"            {relu_recommend['iso3'].nunique()} pays, {relu_recommend['crop'].nunique()} cultures, "
      f"{relu_recommend['year'].min()}-{relu_recommend['year'].max()}, clé unique, 0 NaN")
print("contrôles : OK")

relu_negatifs = pd.read_csv(chemin_negatifs)
assert len(relu_negatifs) == len(negatifs) == 231
assert relu_negatifs.columns.tolist() == df_agri.columns.tolist()
assert (relu_negatifs[CIBLE_PREDICT] < 0).all()

predict   : (999769, 10) | colonnes : ['Crop', 'Soil_Type', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used', 'Region', 'Weather_Condition', 'Days_to_Harvest', 'Yield_tons_per_hectare']
recommend : (15664, 9) | colonnes : ['iso3', 'area', 'year', 'crop', 'temp_hist', 'rain_mm', 'pest_hist', 'log_pest_hist', 'yield_t_ha']
            117 pays, 10 cultures, 1991-2013, clé unique, 0 NaN
contrôles : OK


# Conclusion
- `/predict` : 999 769 lignes, neuf variables d’entrée conservées ; 231 rendements négatifs sauvegardés séparément.
- `/recommend` : 15 664 lignes ; pesticides historiques conservés en brut et en `log1p` pour comparaison.
- `iso3`, `area` et `year` restent disponibles comme contexte hors modèle. 2013 est réservée au test final.

**Limites :** données historiques nationales, conditions inhabituelles à signaler, rendement prédit différent de la rentabilité.